In [3]:
import tensorflow_io as tfio
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Paths
BASE_DIR = "/home/spaulgupta2000/data/pcam_raw"
TRAIN_CSV = os.path.join(BASE_DIR, "split_train_80.csv")   # 80-20 split train labels
VAL_CSV   = os.path.join(BASE_DIR, "split_test_80.csv")    # 80-20 split test labels

TRAIN_SR_DIR = os.path.join(BASE_DIR, "train_full_output")
VAL_SR_DIR   = os.path.join(BASE_DIR, "test_full_output")

# Run name
RUN_NAME = "resnet50_sr_80_run2"

SAVE_DIR = os.path.join(BASE_DIR, f"results_{RUN_NAME}")
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_H5 = os.path.join(SAVE_DIR, f"{RUN_NAME}_best.h5")
FINAL_H5 = os.path.join(SAVE_DIR, f"{RUN_NAME}_final.h5")

NPY_TRUE = os.path.join(SAVE_DIR, f"y_true_{RUN_NAME}.npy")
NPY_PRED = os.path.join(SAVE_DIR, f"y_pred_{RUN_NAME}.npy")
NPY_PROB = os.path.join(SAVE_DIR, f"y_prob_{RUN_NAME}.npy")

HISTORY_JSON = os.path.join(SAVE_DIR, f"history_{RUN_NAME}.json")
METRICS_JSON = os.path.join(SAVE_DIR, f"metrics_{RUN_NAME}.json")

# Settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4

# Fine-tuning options
FINE_TUNE = True
UNFREEZE_LAST_N = 30
# If FINE_TUNE = False -> freeze full ResNet50 base
# If FINE_TUNE = True and UNFREEZE_LAST_N = 30 -> unfreeze last 30 layers
# If FINE_TUNE = True and UNFREEZE_LAST_N = None -> unfreeze all layers

# Augmentation option
USE_AUGMENTATION = True

def load_image(path):
    raw = tf.io.read_file(path)
    img = tfio.experimental.image.decode_tiff(raw)

    # Ensure rank-3 (H, W, C)
    img = tf.ensure_shape(img, [None, None, None])

    c = tf.shape(img)[-1]

    # If grayscale, convert to RGB
    img = tf.cond(
        tf.equal(c, 1),
        lambda: tf.image.grayscale_to_rgb(img),
        lambda: img
    )

    # If >3 channels, keep first 3
    c2 = tf.shape(img)[-1]
    img = tf.cond(
        tf.greater(c2, 3),
        lambda: img[..., :3],
        lambda: img
    )

    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def augment_image(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def make_ds(df, folder, shuffle=False):
    paths = df["id"].astype(str).apply(lambda x: os.path.join(folder, f"{x}.tif")).values
    labels = df["label"].values.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _map(p, y):
        x = load_image(p)
        if shuffle and USE_AUGMENTATION:
            x = augment_image(x)
        return x, y

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(20000, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def build_model():
    base = keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling="avg"
    )

    if not FINE_TUNE:
        base.trainable = False
    else:
        base.trainable = True
        if UNFREEZE_LAST_N is not None:
            for layer in base.layers[:-UNFREEZE_LAST_N]:
                layer.trainable = False
            for layer in base.layers[-UNFREEZE_LAST_N:]:
                layer.trainable = True

    x = keras.layers.Dropout(0.4)(base.output)
    out = keras.layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(base.input, out)
    return model

class EpochMetricsPrinter(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(
            f"Epoch {epoch + 1:03d} | "
            f"loss={logs.get('loss', float('nan')):.6f} | "
            f"accuracy={logs.get('accuracy', float('nan')):.6f} | "
            f"auc={logs.get('auc', float('nan')):.6f} | "
            f"val_loss={logs.get('val_loss', float('nan')):.6f} | "
            f"val_accuracy={logs.get('val_accuracy', float('nan')):.6f} | "
            f"val_auc={logs.get('val_auc', float('nan')):.6f}",
            flush=True
        )

def main():
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)

    print("Train rows:", len(train_df), flush=True)
    print("Val rows:", len(val_df), flush=True)
    print(f"Fine-tune enabled: {FINE_TUNE}", flush=True)
    print(f"Unfreeze last N layers: {UNFREEZE_LAST_N}", flush=True)
    print(f"Use augmentation: {USE_AUGMENTATION}", flush=True)

    train_ds = make_ds(train_df, TRAIN_SR_DIR, shuffle=True)
    val_ds = make_ds(val_df, VAL_SR_DIR, shuffle=False)

    model = build_model()

    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            BEST_H5,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        ),
        EpochMetricsPrinter()
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        initial_epoch=0,
        callbacks=callbacks,
        verbose=0
    )

    model.save(FINAL_H5)

    # Save history JSON
    history_dict = {}
    for k, v in history.history.items():
        history_dict[k] = [float(x) for x in v]

    with open(HISTORY_JSON, "w") as f:
        json.dump(history_dict, f, indent=4)

    # Save predictions on validation set
    y_true_list = []
    y_prob_list = []

    for xb, yb in val_ds:
        probs = model.predict(xb, verbose=0).ravel()
        y_prob_list.append(probs)
        y_true_list.append(yb.numpy().ravel())

    y_prob = np.concatenate(y_prob_list, axis=0)
    y_true = np.concatenate(y_true_list, axis=0).astype(int)
    y_pred = (y_prob >= 0.5).astype(int)

    np.save(NPY_TRUE, y_true)
    np.save(NPY_PRED, y_pred)
    np.save(NPY_PROB, y_prob)

    # Metrics
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    auc_metric = tf.keras.metrics.AUC()
    auc_metric.update_state(y_true, y_prob)
    auc = float(auc_metric.result().numpy())

    metrics = {
        "run_name": RUN_NAME,
        "fine_tune": FINE_TUNE,
        "unfreeze_last_n": UNFREEZE_LAST_N,
        "use_augmentation": USE_AUGMENTATION,
        "best_h5": BEST_H5,
        "final_h5": FINAL_H5,
        "y_true_npy": NPY_TRUE,
        "y_pred_npy": NPY_PRED,
        "y_prob_npy": NPY_PROB,
        "history_json": HISTORY_JSON,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "auc": auc
    }

    with open(METRICS_JSON, "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"✅ Saved best model : {BEST_H5}", flush=True)
    print(f"✅ Saved final model: {FINAL_H5}", flush=True)
    print(f"✅ Saved history    : {HISTORY_JSON}", flush=True)
    print(f"✅ Saved metrics    : {METRICS_JSON}", flush=True)
    print(f"✅ Saved y_true     : {NPY_TRUE}", flush=True)
    print(f"✅ Saved y_pred     : {NPY_PRED}", flush=True)
    print(f"✅ Saved y_prob     : {NPY_PROB}", flush=True)

if __name__ == "__main__":
    main()


Python exe: C:\Users\spgmm\anaconda3\python.exe
Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


![Pipeline](../assets/Resnet50_running.png)
